# ノートブック5：摩擦への頑健性の回復と、提出用の方策の確定（ランタイム：A100・ハイメモリ）

ノートブック4の方策（`model_26997`）は、摩擦 0.5・0.8 では完走したが、**摩擦 1.0 では 7.36 秒で止まった**（`ee_body_pos`）。
手元の解析【実測】では、崩れ始めは **5.5 秒ごろ**で、6.3〜7.36 秒の約 1 秒で立ち位置が 7 cm → **101 cm** 流れていた（摩擦 0.8 では同じ時刻で 9 cm 以下）。
お手本の 4.5〜5.25 秒は約 50° の方向転換、6.5〜7.75 秒は ±35° の速いひねりで、どちらも足が床の高さのまま 0.2〜0.56 m/s で動いている。

このノートブックでやること（上から順に。**どこで止めても、それまでの結果はドライブに残る**）：

| 手順 | 中身 | 目安 |
|---|---|---|
| 2 | **スクリーニング**（＋4b 繰り返し評価）：ノートブック4のチェックポイント 7 個を摩擦 1.0 で評価し、完走したものを 0.5 / 0.8 / 0.9 / 1.1 でも評価する | 実測 52 分（並列 4 本）＋4b で約 15 分 |
| 3 | **摩擦 1.0 の診断**：どの部位がしきい値を超えたか、立ち位置が流れる様子を 0〜9 秒の動画・グラフ・コマで見る | 10〜15 分 |
| 4 | **追加学習（NB5）**：摩擦のばらつき 0.6〜1.0 → **0.4〜1.2**、外乱 0.5 → **0.75 倍**。報酬は NB4 のまま。2,500 反復 | 約 1.2 時間・約 15 CU |
| 5 | **評価と提出用の方策の確定**：摩擦 6 条件で比べ、提出用の 1 つを選んで正面の比較動画を作る | 30〜60 分 |

**事前準備**：手元の `pai_final/scripts/render_policy.py`（10/19 昼の版：`--dump-bodies`・`--t-end`・`--no-ghost`、転倒の 0.02 秒前の部位の記録）を、ドライブの `pai_final/scripts/` に上書きアップロードしておくこと（手順 1 で確認して、古ければ止まる）。

## 1. ドライブの接続と設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

# ▼ ノートブック5の設定
TASK = 'Unitree-G1-Tracking'
NAME = 'the_feels_full_g25s'                                  # お手本（NB4 と同じ。変えると誤差の基準が変わるので変えない）
RUN4 = '2026-10-18_09-20-40_the_feels_full_g25s_ft2'          # ノートブック4の追加学習
FINAL4 = 'model_26997.pt'                                     # ノートブック4の最後のチェックポイント
# スクリーニング
STAGE1_MU = [1.0]                      # 段階1：全チェックポイントをこの摩擦で
STAGE2_MU = [0.5, 0.8, 0.9, 1.1]       # 段階2：段階1で完走したもの（と FINAL4）を、この摩擦でも
ROBUST_MU = [0.5, 0.8, 0.9, 1.0, 1.1]  # 「頑健」と呼ぶ条件：これらすべてで完走
PARALLEL = 4                           # 同時に走らせる評価の数（A100 80GB なら 4 で余裕。メモリ不足なら 2 に）
REPEATS = 4                            # 手順 4b：同じ条件を何回評価するか（評価は毎回少しずつ結果が変わる）
PARALLEL_REP = 6                       # 手順 4b・5 の同時実行数（1 本あたり CPU 1 コア。A100 の Colab は 12 コア）
# 追加学習（NB5）
START_CKPT = 'auto'    # 'auto' = スクリーニングで選ばれた最良の頑健なチェックポイント（なければ FINAL4）。直接 'model_25500.pt' のようにも書ける
ITERS5 = 2500          # 追加の反復回数（A100 で約 1.1〜1.2 時間）
FRIC_MIN, FRIC_MAX, PUSH = 0.4, 1.2, 0.75
ACTION_RATE, BODY_POS_STD, BODY_ORI_STD, ROOT_POS_STD = -0.05, 0.25, 0.3, 0.2   # NB4 と同じ（変えない）
RUN5_NAME = f'{NAME}_ft3_rob'          # run 名（rob = robust）
FINAL_MU = [0.5, 0.8, 0.9, 1.0, 1.1, 1.2]   # 最後の評価の摩擦
# 正面カメラ（NB4 と同じ）
AZ, EL, DIST = 270, -5, 2.8
for k, v in dict(TASK=TASK, NAME=NAME, RUN4=RUN4, FINAL4=FINAL4, ITERS5=ITERS5, FRIC_MIN=FRIC_MIN, FRIC_MAX=FRIC_MAX,
                 PUSH=PUSH, ACTION_RATE=ACTION_RATE, BODY_POS_STD=BODY_POS_STD, BODY_ORI_STD=BODY_ORI_STD,
                 ROOT_POS_STD=ROOT_POS_STD, RUN5_NAME=RUN5_NAME, AZ=AZ, EL=EL, DIST=DIST).items():
    os.environ[k] = str(v)
os.environ['START'] = '0'
L = '/content/unitree_rl_mjlab'
EVAL = f'{P}/outputs/eval/nb5'          # このノートブックの評価結果はすべてここ（既にある結果は再計算しない）
FIGS = f'{P}/outputs/report_figs'
os.makedirs(EVAL, exist_ok=True); os.makedirs(FIGS, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls $P/logs/rsl_rl/g1_tracking/
!ls $P/logs/rsl_rl/g1_tracking/$RUN4/ | grep model_

## 2. unitree_rl_mjlab のインストール（5〜10分）とお手本の配置

In [ ]:
%%bash
set -e
pip -q install uv
bash $P/scripts/pin_clone.sh $URL_unitree_rl_mjlab /content/unitree_rl_mjlab $REV_unitree_rl_mjlab
uv venv /content/venv_mjlab --python 3.11 -q --clear
cd /content/unitree_rl_mjlab
# mjlab 1.2.0（2026年3月公開）は MuJoCo 系の版を固定していないため、そのままだと最新版が入り
# 「mujoco と mujoco-warp の版が合わない」エラーになる。公開当時の組み合わせに固定する。
cat > /content/mjlab_pins.txt <<'EOF'
mujoco==3.5.0
mujoco-warp==3.5.0.2
warp-lang==1.12.0
EOF
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt -e .     # mjlab==1.2.0 が入る
uv pip install --python /content/venv_mjlab/bin/python -q --override /content/mjlab_pins.txt matplotlib scipy   # scipy は mjlab の地形モジュールが使う（依存に書かれていない）
# 学習・変換・評価で使う部品をここでまとめて読み込んで確認（足りなければこのセルで止まる）
/content/venv_mjlab/bin/python - <<'EOF'
import sys, os; sys.path.insert(0, os.getcwd())
import torch, mjlab, mujoco, mujoco_warp, warp, scipy, importlib.metadata as m
import mjlab.entity, mjlab.scene, mjlab.envs, mjlab.rl, mjlab.tasks, src.tasks
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| mjlab', m.version('mjlab'),
      '| mujoco', mujoco.__version__, '| mujoco-warp', m.version('mujoco-warp'), '| warp', warp.__version__)
print('[ok] 必要な部品はすべて読み込めました')
EOF
# 学習ログはローカルに置き、ドライブへは監視セルが同期する（ドライブにあれば戻す）
mkdir -p logs/rsl_rl $P/logs/rsl_rl
bash $P/scripts/sync_logs.sh $P/logs/rsl_rl logs/rsl_rl
ls logs/rsl_rl/*/ 2>/dev/null | tail -5 || true
# スクリプトが新しい版か確認（古い版のまま動かすと、診断の列が出ない）
grep -q -- "--dump-bodies" $P/scripts/render_policy.py || { echo "[stop] render_policy.py が古い版です。手元の版をドライブに上書きしてください"; exit 1; }
grep -q -- "--no-ghost" $P/scripts/render_policy.py || { echo "[stop] render_policy.py に --no-ghost がありません。10/19 の版をドライブに上書きしてください"; exit 1; }
grep -q "z_prev" $P/scripts/render_policy.py || { echo "[stop] render_policy.py が 10/19 昼の修正（転倒の部位の記録）より前の版です。手元の版をドライブに上書きしてください"; exit 1; }
grep -q -- "--friction-min" $P/scripts/train_sim.py || { echo "[stop] train_sim.py が古い版です"; exit 1; }
# お手本（NB4 で作った npz）を配置
cp $P/outputs/motions/$NAME.npz src/assets/motions/g1/
ls -la src/assets/motions/g1/ | grep $NAME
ls logs/rsl_rl/g1_tracking/$RUN4/ | grep model_ || { echo "[stop] NB4 のチェックポイントが見つかりません"; exit 1; }

## 3. 評価の共通関数

`render_policy.py` を別のプロセスで呼び、結果の JSON を読む。**同じ名前の結果がドライブにあれば計算し直さない**ので、切断されても続きから再開できる。
複数の評価を同時に走らせる（`PARALLEL`）。評価は環境 1 つで、GPU をほとんど使わないため、同時に走らせたほうが全体は速い。

In [ ]:
import subprocess, json, time, glob, re, concurrent.futures as cf
import numpy as np, pandas as pd
PY = '/content/venv_mjlab/bin/python'
KEYS = ['success', 'first_fall_s', 'fall_reason', 'fall_bodies', 'mean_joint_err_rad', 'mean_body_pos_err_rel_m',
        'max_body_pos_err_rel_m', 'max_root_xy_drift_m', 'mean_action_rate']

def ckpt_path(run, ckpt):
    return f"{L}/logs/rsl_rl/g1_tracking/{run}/{ckpt}"

def tag_of(run, ckpt, mu, suffix=''):
    return f"{run.split('_', 2)[-1]}_{ckpt.replace('.pt', '')}_mu{mu}{suffix}"

def evaluate(run, ckpt, mu, suffix='', extra=(), video=False, force=False):
    """1 回の評価。結果の dict を返す（転倒しても例外にはしない）"""
    tag = tag_of(run, ckpt, mu, suffix)
    out = f"{EVAL}/{tag}.mp4"; js = out[:-4] + '.json'
    if os.path.exists(js) and not force:
        j = json.load(open(js)); j['_tag'] = tag; j['_cached'] = True; return j
    ck = ckpt_path(run, ckpt)
    if not os.path.exists(ck):
        raise FileNotFoundError(ck)
    cmd = [PY, f'{P}/scripts/render_policy.py', '--task', TASK, '--motion-file', f'src/assets/motions/g1/{NAME}.npz',
           '--checkpoint', ck, '--friction', str(mu), '--out', out, *extra]
    if not video:
        cmd.append('--no-video')
    t0 = time.time()
    r = subprocess.run(cmd, cwd=L, capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(js):
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise RuntimeError(f'評価に失敗しました: {tag}')
    j = json.load(open(js)); j['_tag'] = tag; j['_sec'] = round(time.time() - t0); j['_cached'] = False
    return j

def evaluate_many(jobs, parallel=None):
    """jobs = [(run, ckpt, mu[, suffix[, extra[, video]]]), ...] を PARALLEL 本ずつ同時に評価する"""
    out = []
    with cf.ThreadPoolExecutor(max_workers=parallel or PARALLEL) as ex:
        futs = {ex.submit(evaluate, *job): job for job in jobs}
        for f in cf.as_completed(futs):
            run, ck, mu = futs[f][:3]
            suffix = futs[f][3] if len(futs[f]) > 3 else ''
            j = f.result()
            row = dict(run=run.split('_', 2)[-1], ckpt=ck, mu=mu, rep=suffix, tag=j['_tag'], **{k: j.get(k) for k in KEYS})
            out.append(row)
            took = '（保存済み）' if j.get('_cached') else f"（{j.get('_sec')} 秒）"
            print(f"  {ck:<16}{suffix:<6} 摩擦 {mu:<4} 完走={str(j['success']):<5} 転倒={str(j['first_fall_s']):<6} "
                  f"関節={j['mean_joint_err_rad']:<7} 位置={j['mean_body_pos_err_rel_m']:<7} ドリフト最大={j['max_root_xy_drift_m']} {took}")
    return out

def ckpts_of(run):
    cs = [os.path.basename(c) for c in glob.glob(ckpt_path(run, 'model_*.pt'))]
    return sorted(cs, key=lambda s: int(re.findall(r'\d+', s)[0]))

print('NB4 のチェックポイント:', ckpts_of(RUN4))

## 4. スクリーニング：摩擦 0.5〜1.1 のすべてで完走するチェックポイントを探す

- **段階1**：7 個すべてを摩擦 1.0 で評価（転倒すればそこで止まるので速い）
- **段階2**：1.0 で完走したもの（と、比較のため `model_26997`）を 0.5 / 0.8 / 0.9 / 1.1 でも評価
- 0.9 と 1.1 を足すのは、1.0 だけの**偶然の失敗／偶然の成功**を見分けるため。評価は毎回同じ結果になる（決定論的）ので、「1 回完走した」だけでは、境界ぎりぎりかどうかが分からない。摩擦を少しずつ変えるのが、ばらつきを見る代わりになる
- **頑健** ＝ `ROBUST_MU` のすべてで完走。その中で、摩擦 0.8 の関節誤差が一番小さいものを「最良」とする

**最初の確認**：`model_26997` の摩擦 0.8 の関節誤差が、NB4 の値（0.0943 rad）とほぼ同じになるか（GPU の計算順の違いで小数第 4 位は揺れる）。

In [ ]:
t0 = time.time()
C4 = ckpts_of(RUN4)
print('[段階1] すべてのチェックポイントを摩擦', STAGE1_MU)
rows = evaluate_many([(RUN4, c, mu) for c in C4 for mu in STAGE1_MU])
pass1 = sorted({r['ckpt'] for r in rows if r['success']}, key=lambda s: int(re.findall(r'\d+', s)[0]))
print(f"\n摩擦 {STAGE1_MU} で完走: {pass1 or 'なし'}")
cand = pass1 + ([FINAL4] if FINAL4 not in pass1 else [])
print('\n[段階2]', cand, '× 摩擦', STAGE2_MU)
rows += evaluate_many([(RUN4, c, mu) for c in cand for mu in STAGE2_MU])
df = pd.DataFrame(rows)
df.to_csv(f'{EVAL}/screen_nb4.csv', index=False)
print(f'\n所要 {time.time() - t0:.0f} 秒')

# 表：行 = チェックポイント、列 = 摩擦。○ = 完走、数字 = 転倒した時刻
tab = df.assign(r=df.apply(lambda r: '○' if r.success else f"×{r.first_fall_s}", axis=1)).pivot_table(
    index='ckpt', columns='mu', values='r', aggfunc='first')
tab = tab.reindex(sorted(tab.index, key=lambda s: int(re.findall(r'\d+', s)[0])))
print('\n完走の表（○ = 完走、×数字 = 止まった時刻 [秒]）'); display(tab)
m08 = df[df.mu == 0.8].set_index('ckpt')[['mean_joint_err_rad', 'mean_body_pos_err_rel_m', 'max_body_pos_err_rel_m', 'max_root_xy_drift_m', 'mean_action_rate']]
print('摩擦 0.8 の誤差'); display(m08)

ok = df.groupby('ckpt').apply(lambda g: all(bool(g[g.mu == mu].success.any()) for mu in ROBUST_MU))
robust = [c for c in ok.index if ok[c]]
BEST4 = min(robust, key=lambda c: m08.loc[c, 'mean_joint_err_rad']) if robust else None
print(f"\n頑健（{ROBUST_MU} すべて完走）: {robust or 'なし'}")
print(f"最良の頑健なチェックポイント: {BEST4}")
open(f'{EVAL}/best_nb4.txt', 'w').write(str(BEST4))

**読み方**

`model_24000` は NB4 の学習を始めて 2 反復目の版で、中身はほぼ NB3 の方策（`model_23998`）と同じ。

| 結果 | 意味 | 次の手 |
|---|---|---|
| 頑健なものがあり、誤差が `model_26997` とほぼ同じ | 学習の途中に「正確さを保ったまま頑健」な版がある | それを NB5 の出発点にする（`START_CKPT='auto'`）。NB5 が失敗しても提出に使える |
| `model_26997` が 0.9・1.1 では完走 | 1.0 だけの境界ぎりぎりの失敗（決定論的な評価の偶然） | 致命的ではない。NB5 で余裕を持たせる |
| `model_26997` が 1.1 でも止まる | 高い摩擦に対して系統的に弱い（仮説「足の引っかかり」に合う） | NB5 の摩擦の範囲を広げる効果が大きいはず |
| どれも 1.0 で止まる | NB4 の学習全体で弱くなっていた | NB3（`model_23998`、1.0 で完走）を予備にしつつ NB5 |

## 4b. 繰り返し評価：同じ条件でも結果が変わる

**10/19 の実行で分かったこと【実測】**：`model_26997`・摩擦 1.0 は、NB4 の評価では 7.36 秒、手順 4 では **6.56 秒**で止まり、手順 5 の診断用の実行では**最後まで完走**した（関節 0.0951 rad）。
評価は「摩擦を固定し、ランダム化を外す」ので毎回同じになるはずだったが、GPU の物理計算（MuJoCo Warp）は足し算の順番が毎回わずかに違う。ふだんは小数第 4 位の揺れ（0.0943 / 0.0944）で済むが、**境界ぎりぎりの場面では、その小さな差が「止まる／止まらない」を分ける**。

したがって「1 回完走した」は頑健さの証拠として弱い。手順 4 で頑健と判定されたうち、摩擦 0.8 の関節誤差が小さい上位 2 つと `model_26997` を、摩擦 1.0 で `REPEATS` 回ずつ評価して、**完走の回数**で比べる。

In [ ]:
t0 = time.time()
m08 = df[df.mu == 0.8].set_index('ckpt').mean_joint_err_rad
top = sorted(robust, key=lambda c: m08[c])[:2] if robust else []
cand_rep = top + ([FINAL4] if FINAL4 not in top else [])
print('繰り返し評価:', cand_rep, f'× 摩擦 1.0 × {REPEATS} 回')
rrows = evaluate_many([(RUN4, c, 1.0, f'_rep{k}') for c in cand_rep for k in range(1, REPEATS + 1)], parallel=PARALLEL_REP)
dr = pd.DataFrame(rrows); dr.to_csv(f'{EVAL}/repeat_nb4_mu1.0.csv', index=False)
print(f'所要 {time.time() - t0:.0f} 秒')
# 手順 4 の 1 回目（摩擦 1.0）も数に入れる
first = df[df.mu == 1.0][['ckpt', 'success', 'first_fall_s']]
allr = pd.concat([dr[['ckpt', 'success', 'first_fall_s']], first[first.ckpt.isin(cand_rep)]])
summ = allr.groupby('ckpt').agg(完走=('success', 'sum'), 回数=('success', 'size'),
                                止まった時刻=('first_fall_s', lambda x: sorted(v for v in x if v is not None and v == v)))
summ['摩擦0.8の関節誤差'] = [m08[c] for c in summ.index]
display(summ)
full = [c for c in summ.index if summ.loc[c, '完走'] == summ.loc[c, '回数']]
BEST4 = min(full, key=lambda c: m08[c]) if full else (BEST4 if BEST4 else None)
print(f"摩擦 1.0 で毎回完走: {full or 'なし'} → NB5 の出発点の候補（最良の頑健なチェックポイント）: {BEST4}")
open(f'{EVAL}/best_nb4.txt', 'w').write(str(BEST4))

## 5. 摩擦 1.0 の診断：止まるときに何が起きているか

止まるかどうかが実行ごとに変わるので（4b）、**同じ条件を何回か撮り、立ち位置が一番流れた 1 本を「失敗の例」として見る**。

1. 終了条件ありの評価（4b）で止まった回の `fall_bodies`（**転倒の 0.02 秒前**に高さのずれが大きかった部位）を表にする。10/19 昼に直した `render_policy.py` が必要（それより前の版は、自動リセットの後の値を記録していた）
2. 終了条件なしで 0〜9 秒だけを、摩擦 1.0 で `REPEATS` 回、摩擦 0.8 で 1 回撮る（`--t-end 9 --dump-bodies`、動画つき）
3. 摩擦 1.0 で 0〜9 秒の最大ドリフトが一番大きい回と、摩擦 0.8 を、4〜9 秒で左右に並べる。グラフとコマ（5.0〜7.4 秒）も作る

**見るところ**：5.5 秒以降、失敗の回だけ**足が床に止まったまま体が回る／流れる**か（仮説：足の引っかかり）、足を余計に踏み出して離れていくか。

In [ ]:
D = FINAL4
# (1) 止まった回の部位（4b の結果。古い版の render_policy.py で作った結果は値が意味をなさない）
fb = dr[(dr.ckpt == D) & (dr.success == False)][['rep', 'first_fall_s', 'fall_bodies']] if 'dr' in dir() else None
print('止まった回と、転倒の 0.02 秒前に高さのずれが大きかった部位（お手本 − ロボット [m]、しきい値 0.25）:')
display(fb)
# (2) 終了条件なしで 0〜9 秒（動画つき）。RAM は 1 本 2〜3 GB 程度
jobs = [(RUN4, D, 1.0, f'_noterm9_r{k}', ('--no-terminations', '--t-end', '9', '--dump-bodies'), True) for k in range(1, REPEATS + 1)]
jobs += [(RUN4, D, 0.8, '_noterm9', ('--no-terminations', '--t-end', '9', '--dump-bodies'), True)]
nrows = pd.DataFrame(evaluate_many(jobs, parallel=PARALLEL_REP))
display(nrows[['mu', 'rep', 'max_root_xy_drift_m', 'mean_joint_err_rad', 'mean_body_pos_err_rel_m']].sort_values(['mu', 'rep']))
w = nrows[nrows.mu == 1.0].sort_values('max_root_xy_drift_m').iloc[-1]
FAIL_TAG, OK_TAG = w.tag, nrows[nrows.mu == 0.8].iloc[0].tag
print(f"失敗の例として使う回: {FAIL_TAG}（0〜9 秒の最大ドリフト {w.max_root_xy_drift_m} m）/ 比較: {OK_TAG}")
if w.max_root_xy_drift_m < 0.3:
    print('[info] この回数では、摩擦 1.0 で大きく流れる回が出なかった。REPEATS を増やすか、そのまま次へ進んでよい（失敗は確率的にしか起きない）')

In [ ]:
# 4〜9 秒を左右に並べる（左：摩擦 1.0 の失敗の例、右：摩擦 0.8）。bash を使わず Python から ffmpeg を呼ぶ
OUTV = f"{EVAL}/diag_mu1.0_vs_0.8_4to9s.mp4"
A, B = f"{EVAL}/{FAIL_TAG}.mp4", f"{EVAL}/{OK_TAG}.mp4"
lab = ("[0:v]drawtext=text='friction 1.0':x=20:y=20:fontsize=40:fontcolor=white:box=1:boxcolor=black@0.5[a];"
       "[1:v]drawtext=text='friction 0.8':x=20:y=20:fontsize=40:fontcolor=white:box=1:boxcolor=black@0.5[b];[a][b]hstack")
base_cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-ss', '4', '-t', '5', '-i', A, '-ss', '4', '-t', '5', '-i', B]
enc = ['-c:v', 'libx264', '-crf', '23', '-pix_fmt', 'yuv420p', OUTV]
r = subprocess.run(base_cmd + ['-filter_complex', lab] + enc, capture_output=True, text=True, stdin=subprocess.DEVNULL)
if r.returncode != 0:   # 文字入れ（drawtext）が使えない環境では、文字なしで並べる（左：1.0、右：0.8）
    print('[info] 文字入れができないため、文字なしで並べます（左：摩擦 1.0、右：摩擦 0.8）')
    r = subprocess.run(base_cmd + ['-filter_complex', '[0:v][1:v]hstack'] + enc, capture_output=True, text=True, stdin=subprocess.DEVNULL)
if r.returncode != 0:
    print(r.stderr[-2000:]); raise RuntimeError('比較動画を作れませんでした')
print(OUTV, f"{os.path.getsize(OUTV) / 1e6:.1f} MB")
display(show(OUTV, width=1100))

In [ ]:
# グラフ：ドリフト・部位の高さのずれ・足の水平速度（失敗の例と摩擦 0.8）
import matplotlib.pyplot as plt, cv2   # 図の文字は英語（Colab には日本語フォントがないため）
fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for tag, name, ls in ((FAIL_TAG, 'friction 1.0', '-'), (OK_TAG, 'friction 0.8', '--')):
    e = pd.read_csv(f"{EVAL}/{tag}.csv"); b = pd.read_csv(f"{EVAL}/{tag}_bodies.csv")
    ax[0].plot(e.time_s, e.root_xy_drift_m * 100, ls, label=name)
    for c in [c for c in b.columns if c.startswith('zerr_') and ('wrist' in c or 'ankle' in c)]:
        ax[1].plot(b.time_s, b[c], ls, lw=1, label=f"{c[5:]} ({name})")
    for c in [c for c in b.columns if c.startswith('robot_xyspeed_')]:
        ax[2].plot(b.time_s, b[c], ls, lw=1, label=f"{c[14:]} ({name})")
ax[0].set_ylabel('root xy drift [cm]'); ax[0].legend()
ax[1].axhline(0.25, c='r', lw=0.8); ax[1].axhline(-0.25, c='r', lw=0.8); ax[1].set_ylabel('height error [m]\n(ref - robot)'); ax[1].legend(fontsize=7, ncol=2)
ax[2].set_ylabel('robot foot\nxy speed [m/s]'); ax[2].set_xlabel('time [s]'); ax[2].legend(fontsize=7, ncol=2)
for a_ in ax:
    a_.axvspan(4.5, 5.25, color='orange', alpha=0.15); a_.axvspan(6.5, 7.75, color='orange', alpha=0.15); a_.set_xlim(3, 9)
fig.suptitle('friction 1.0 failure example (orange: turn 4.5-5.25 s, twist 6.5-7.75 s in the reference)')
plt.tight_layout(); plt.savefig(f'{FIGS}/fig5_friction_diag.png', dpi=130); plt.show()

# コマ：5.0〜7.4 秒（上：失敗の例、下：摩擦 0.8）
ts = [5.0, 5.5, 6.0, 6.5, 7.0, 7.36]
fig, ax = plt.subplots(2, len(ts), figsize=(3.2 * len(ts), 4.4))
for r_, (tag, name) in enumerate(((FAIL_TAG, 'friction 1.0'), (OK_TAG, 'friction 0.8'))):
    cap = cv2.VideoCapture(f"{EVAL}/{tag}.mp4")
    for c, t in enumerate(ts):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(t * 50) - 1); ok_, im = cap.read()   # k 枚目のコマ = (k+1)/50 秒
        if ok_: ax[r_, c].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax[r_, c].axis('off'); ax[r_, c].set_title(f'{name} / {t} s', fontsize=9)
    cap.release()
plt.tight_layout(); plt.savefig(f'{FIGS}/fig5b_friction_frames.png', dpi=110); plt.show()

## 6. 追加学習（NB5）：摩擦 0.4〜1.2・外乱 0.75 倍・2,500 反復

- 出発点：`START_CKPT='auto'` なら、手順 4 の最良の頑健なチェックポイント（なければ `model_26997`）
- **変えるのは「摩擦の範囲」と「外乱」の 2 つだけ**。報酬は NB4 のまま。お手本も NB4 と同じ（32 秒付近は手元の解析で「お手本の欠陥」とは言えなかったため、補正しない）
- ログの先頭の `[train_sim]` 行に **`足裏摩擦 (0.4, 1.2) / 外乱(x方向の速度) (-0.375, 0.375)`** と `motion_global_root_pos std 0.2` が出ていることを確認する
- 範囲を広げた直後の数百反復は、エピソード長と報酬が一時的に下がるのが普通（**最初の 500 反復で判断しない**）
- 行動の std（`Mean action std`）が 0.2 を切ったら、それ以上は続けない（NB4 の最後で 0.24）

In [ ]:
if START_CKPT == 'auto':
    b = open(f'{EVAL}/best_nb4.txt').read().strip() if os.path.exists(f'{EVAL}/best_nb4.txt') else 'None'
    START = b if b != 'None' else FINAL4
else:
    START = START_CKPT
os.environ['START_CKPT_RESOLVED'] = START
print('NB5 の出発点:', RUN4, '/', START)

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
if [ -f /content/train.pid ] && [ -d /proc/$(cat /content/train.pid) ]; then
  echo "[stop] 学習がすでに動いています（PID $(cat /content/train.pid)）。"; exit 1
fi
# 【10/19 追加】NB5 の学習が済んでいれば（ドライブに run とチェックポイントがあれば）始めない。「すべてのセルを実行」で学習が二重に走るのを防ぐ
DONE=$(ls -d logs/rsl_rl/g1_tracking/*_${RUN5_NAME} 2>/dev/null | head -1)
if [ -n "$DONE" ] && ls $DONE/model_*.pt >/dev/null 2>&1; then
  echo "[skip] NB5 の学習は済んでいます: $(basename $DONE)（最後: $(ls -v $DONE/model_*.pt | tail -1 | xargs basename)）。学習し直すときは RUN5_NAME を変える"
  rm -f /content/train.pid; exit 0
fi
[ -f logs/rsl_rl/g1_tracking/$RUN4/$START_CKPT_RESOLVED ] || { echo "[stop] 読み込むチェックポイントがありません: $RUN4/$START_CKPT_RESOLVED"; exit 1; }
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz \
  --env.scene.num-envs 4096 \
  --agent.max-iterations $ITERS5 \
  --agent.logger tensorboard \
  --agent.run-name $RUN5_NAME \
  --agent.resume True --agent.load-run "$RUN4" --agent.load-checkpoint "$START_CKPT_RESOLVED" \
  --friction-min $FRIC_MIN --friction-max $FRIC_MAX --push-scale $PUSH \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD \
  --root-pos-std $ROOT_POS_STD \
  > $P/logs/train_$RUN5_NAME.log 2>&1 &
echo $! > /content/train.pid
echo $ITERS5 > /content/train.iters
sleep 90; grep -E "train_sim|Loading model|Error|error" $P/logs/train_$RUN5_NAME.log | head
echo "学習を開始しました（PID $(cat /content/train.pid)）。ログ: $P/logs/train_$RUN5_NAME.log"

In [ ]:
# 監視セル：学習が終わるまで実行したままにする（止めても学習は止まらない）
import re, subprocess, time
from IPython.display import clear_output
# 【10/19 追加】学習を始めていない（上のセルがスキップした）ときは、何もしない
if not os.path.exists('/content/train.pid'):
    print('[skip] 動いている学習はありません（上のセルでスキップ）。手順 7 へ進みます')
else:
  pid = int(open('/content/train.pid').read())
  log = f"{P}/logs/train_{RUN5_NAME}.log"
  alive = lambda: os.path.exists(f"/proc/{pid}")
  sync = lambda: subprocess.run(['bash', f'{P}/scripts/sync_logs.sh', f'{L}/logs/rsl_rl', f'{P}/logs/rsl_rl'])
  last = time.time()
  while alive():
      time.sleep(60)
      txt = open(log, errors='ignore').read().splitlines()
      keep = [l.strip() for l in txt if re.search(r'Learning iteration|Mean episode length|Mean reward|Mean action std|ee_body_pos|ETA|Error|error', l)]
      clear_output(wait=True)
      print(time.strftime('%H:%M:%S'), f'学習中 (PID {pid})'); print('\n'.join(keep[-7:]))
      if time.time() - last > 600:
          sync(); last = time.time(); print('[sync] ドライブへ同期しました')
  sync()
  txt = open(log, errors='ignore').read()
  print('学習プロセスが終了しました。ログの最後:'); print('\n'.join(txt.splitlines()[-15:]))

  # 学習が正常に終わったかを確認（失敗していたら、ここで止めて後ろのセルを実行させない）
  import glob
  runs = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{RUN5_NAME}*"), key=os.path.getmtime)
  ckpts = glob.glob(f"{runs[-1]}/model_*.pt") if runs else []
  its = [int(m) for m in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
  want = int(open('/content/train.iters').read()) if os.path.exists('/content/train.iters') else None
  print(f"[check] run: {runs[-1] if runs else 'なし'} / チェックポイント {len(ckpts)} 個 / 最後の反復 {max(its) if its else 'なし'} / 予定 {want}")
  if 'Traceback' in txt or not ckpts:
      raise RuntimeError('学習が途中でエラー終了しました。上のログの最後を確認してください（後ろのセルは実行しません）。')
  if want and its and max(its) < want * 0.95:
      raise RuntimeError(f'学習が予定の反復数まで進まずに終わりました（{max(its)}/{want}）。ログを確認してください。')
  print('[ok] 学習は正常に終わりました。評価と動画の書き出しに進みます。')

### 切断されたとき

1・2 を実行し直し、次のセルの `REMAIN` に残りの反復回数を入れて実行してから、監視セルをもう一度実行します（再開した run は `_resume` が付き、評価は一番新しい run を使います）。

In [ ]:
%%bash
REMAIN=0      # ← 残りの反復回数を入れたときだけ実行される
if [ "$REMAIN" = "0" ]; then echo "[skip] 再開しません（REMAIN=0）"; exit 0; fi
cd /content/unitree_rl_mjlab
RUN=$(basename $(ls -td logs/rsl_rl/g1_tracking/*_${RUN5_NAME}* | head -1))
echo "再開する run: $RUN / $(ls -v logs/rsl_rl/g1_tracking/$RUN/model_*.pt | tail -1)"
setsid nohup /content/venv_mjlab/bin/python $P/scripts/train_sim.py $TASK \
  --motion-file src/assets/motions/g1/$NAME.npz --env.scene.num-envs 4096 \
  --agent.max-iterations $REMAIN --agent.logger tensorboard --agent.run-name ${RUN5_NAME}_resume \
  --agent.resume True --agent.load-run "$RUN" \
  --friction-min $FRIC_MIN --friction-max $FRIC_MAX --push-scale $PUSH \
  --action-rate-weight $ACTION_RATE --body-pos-std $BODY_POS_STD --body-ori-std $BODY_ORI_STD --root-pos-std $ROOT_POS_STD \
  > $P/logs/train_${RUN5_NAME}_resume.log 2>&1 &
echo $! > /content/train.pid; echo $REMAIN > /content/train.iters; echo started

## 7. 評価：NB5 の方策を摩擦 6 条件で、NB4 と並べて比べる

- NB5 の**最後の 3 つのチェックポイント**を、摩擦 0.5 / 0.8 / 0.9 / 1.0 / 1.1 / 1.2 で評価する
- 比べる相手：NB4 の `model_26997`（摩擦 0.8 の精度の基準）と、手順 4 の最良の頑健なチェックポイント
- 区間別の誤差も出す：**しゃがみ 9〜14 秒**、**方向転換とひねり 4〜8 秒**、**32 秒付近 31〜34 秒**、**10〜30 秒**

**採用の目安**（提出用の方策の選び方）

1. `ROBUST_MU` のすべてで完走すること（必須）
2. 摩擦 0.8 の関節誤差が NB4（0.0943 rad）より **3% 以上悪くならない**こと（目安 0.097 rad 以下）。頑健さと引き換えに精度が少し落ちるのは想定内
3. 1 と 2 を満たすものの中で、摩擦 0.8 の関節誤差が一番小さいもの。なければ、手順 4 の最良の頑健なチェックポイント。それもなければ NB4 の `model_26997`（摩擦 1.0 の失敗はレポートに書く）

In [ ]:
runs5 = sorted(glob.glob(f"{L}/logs/rsl_rl/g1_tracking/*_{RUN5_NAME}*"), key=os.path.getmtime)
assert runs5, 'NB5 の run が見つかりません'
RUN5 = os.path.basename(runs5[-1])
C5 = ckpts_of(RUN5)[-3:]
print('NB5:', RUN5, C5)
jobs = [(RUN5, c, mu) for c in C5 for mu in FINAL_MU]
BEST4 = open(f'{EVAL}/best_nb4.txt').read().strip() if os.path.exists(f'{EVAL}/best_nb4.txt') else 'None'
ref4 = [FINAL4] + ([BEST4] if BEST4 not in ('None', FINAL4) else [])
jobs += [(RUN4, c, mu) for c in ref4 for mu in FINAL_MU]
rows5 = evaluate_many(jobs)
d5 = pd.DataFrame(rows5); d5.to_csv(f'{EVAL}/final_compare.csv', index=False)

d5['id'] = d5.run.str.replace('the_feels_full_g25s_', '') + '/' + d5.ckpt.str.replace('.pt', '')
tab = d5.assign(r=d5.apply(lambda r: '○' if r.success else f"×{r.first_fall_s}", axis=1)).pivot_table(index='id', columns='mu', values='r', aggfunc='first')
print('完走の表'); display(tab)

# 区間別の誤差（摩擦 0.8）
SEGS = {'全編': (0, 999), '4-8秒 ひねり': (4, 8), '9-14秒 しゃがみ': (9, 14), '10-30秒': (10, 30), '31-34秒': (31, 34)}
def seg_table(mu=0.8):
    out = []
    for _, r in d5[d5.mu == mu].iterrows():
        run = RUN5 if r.run == RUN5.split('_', 2)[-1] else RUN4
        a = pd.read_csv(f"{EVAL}/{tag_of(run, r.ckpt, mu)}.csv")
        row = {'id': r.id, '完走': r.success, '最大位置[cm]': round(r.max_body_pos_err_rel_m * 100, 2),
               '最大ドリフト[cm]': round(r.max_root_xy_drift_m * 100, 1), 'act変化': r.mean_action_rate}
        for k, (lo, hi) in SEGS.items():
            m = (a.time_s >= lo) & (a.time_s < hi)
            row[f'{k} 関節'] = round(a.joint_err_rad[m].mean(), 4); row[f'{k} 位置[cm]'] = round(a.body_pos_err_rel_m[m].mean() * 100, 2)
        out.append(row)
    return pd.DataFrame(out).set_index('id')
st = seg_table(0.8); st.to_csv(f'{EVAL}/final_segments_mu0.8.csv'); print('摩擦 0.8 の区間別の誤差'); display(st)

# 採用
ok = d5.groupby('id').apply(lambda g: all(bool(g[g.mu == mu].success.any()) for mu in ROBUST_MU))
j08 = d5[d5.mu == 0.8].set_index('id').mean_joint_err_rad
base = j08.get(f"ft2/{FINAL4[:-3]}", 0.0943)
good = [i for i in ok.index if ok[i] and j08[i] <= base * 1.03]
pick = min(good, key=lambda i: j08[i]) if good else (min([i for i in ok.index if ok[i]], key=lambda i: j08[i]) if ok.any() else f"ft2/{FINAL4[:-3]}")
FINAL_RUN = RUN5 if pick.startswith('ft3') else RUN4
FINAL_CKPT = pick.split('/')[-1] + '.pt'
print(f"\n頑健: {[i for i in ok.index if ok[i]]}\n精度の条件も満たす: {good}\n→ 提出用: {FINAL_RUN} / {FINAL_CKPT}")
open(f'{EVAL}/final_choice.txt', 'w').write(f"{FINAL_RUN}/{FINAL_CKPT}\n")

In [ ]:
# 学習曲線：NB5 のエピソード長・ee_body_pos による終了・行動の std
import os, re, matplotlib.pyplot as plt
txt = open(f"{P}/logs/train_{RUN5_NAME}.log", errors='ignore').read()
def series(key):
    return [float(x) for x in re.findall(rf'{re.escape(key)}:\s+([-\d.]+)', txt)]
it = [int(x) for x in re.findall(r'Learning iteration\s+(\d+)\s*/', txt)]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for a, k in zip(ax, ['Mean episode length', 'Episode_Termination/ee_body_pos', 'Mean action std']):
    s = series(k); a.plot(it[:len(s)], s); a.set_title(k); a.set_xlabel('iteration')
plt.tight_layout(); plt.savefig(f'{FIGS}/fig6_nb5_training.png', dpi=120); plt.show()

## 8. 提出用の方策で、正面の動画を 2 種類（緑のゴーストあり／なし）

手順 7 で選んだ `FINAL_RUN / FINAL_CKPT` を使う。**別のチェックポイントにしたいときは、次のセルの `FINAL_OVERRIDE` だけを書き換える**（例はセルの中）。
左の人のメッシュと中央のお手本は、NB4 で作ったものをそのまま使う。ファイル名にはチェックポイントの名前（`<tag>` ＝ 例 `ft3_rob_model_28500`）が入るので、選び直しても古い動画と混ざらない。

| 出力（`outputs/videos/`） | 中身 | 用途 |
|---|---|---|
| `front_policy_final_<tag>.mp4` | ロボット＋**緑のゴースト**（お手本を半透明で重ねる） | 追従のずれが見える。比較用 |
| `front_policy_final_<tag>_noghost.mp4` | **ロボットだけ**（`--no-ghost`） | 提出用の素材 |
| `comparison_final_<tag>_front_all.mp4` | 3 画面（人／お手本／ロボット＋ゴースト）＋音声 | 比較用（NB4 と同じ形式） |
| `comparison_final_<tag>_front_all_noghost.mp4` | 3 画面（右がロボットだけ）＋音声 | 提出用 |
| `final_robot_only_<tag>_noghost_audio.mp4` | ロボットだけの 1 画面＋元動画の音声 | 提出用（単独で見せるとき） |

- `--no-ghost` は描画の設定（`commands["motion"].debug_vis = False`）だけで、物理計算は変わらない【確認済み：mjlab v1.2.0】。下のセルで、2 本の評価値が同じかも確かめる。
- すでにある動画は作り直さない（作り直したいときは、そのファイルを消してから実行する）。全編の動画は 1 本あたり約 19 GB の RAM を使うので、順番に書き出す。
- **【10/19 修正】** `%%bash` はセルの中身を標準入力から bash に渡す。ffmpeg が標準入力を読むと、セルの残りの行を飲み込んで止まる（`Enter command: …` / `Parse error`）。外部コマンドはすべて `< /dev/null`（ffmpeg は `-nostdin`）で切ってある。

In [ ]:
# 提出用のチェックポイント。None なら手順 7 の自動選定（final_choice.txt）を使う
# 【10/19 決定】提出用は NB4 の途中の版 model_26500（摩擦 0.5〜1.2 の 6 条件すべて完走、摩擦 1.0 は 6 回とも完走）
FINAL_OVERRIDE = ('2026-10-18_09-20-40_the_feels_full_g25s_ft2', 'model_26500.pt')
# FINAL_OVERRIDE = None   # ← 手順 7 の自動選定に戻すとき
# FINAL_OVERRIDE = ('2026-10-19_05-06-11_the_feels_full_g25s_ft3_rob', 'model_28500.pt')   # NB5（手順 7 の自動選定）
if FINAL_OVERRIDE:
    FINAL_RUN, FINAL_CKPT = FINAL_OVERRIDE
else:
    FINAL_RUN, FINAL_CKPT = open(f'{EVAL}/final_choice.txt').read().strip().split('/')
FINAL_TAG = f"{FINAL_RUN.split('_g25s_')[-1]}_{FINAL_CKPT[:-3]}"
os.environ['FINAL_RUN'], os.environ['FINAL_CKPT'], os.environ['FINAL_TAG'] = FINAL_RUN, FINAL_CKPT, FINAL_TAG
assert os.path.exists(f"{L}/logs/rsl_rl/g1_tracking/{FINAL_RUN}/{FINAL_CKPT}"), 'チェックポイントが見つかりません'
print('提出用:', FINAL_RUN, FINAL_CKPT, '/ ファイル名の tag:', FINAL_TAG)

In [ ]:
%%bash
set -e
cd /content/unitree_rl_mjlab
PY=/content/venv_mjlab/bin/python
HUMAN=$P/outputs/gvhmr/the_feels_full/2_global_front.mp4
REF=$P/outputs/motions/${NAME}_retarget_preview_front.mp4
AUDIO=$P/input/the_feels_full.mp4
[ -f $HUMAN ] || { echo "[stop] $HUMAN がありません（ノートブック 04b）"; exit 1; }
[ -f $REF ] || { echo "[stop] $REF がありません（ノートブック 4 の手順 3）"; exit 1; }
[ -n "$FINAL_TAG" ] || { echo "[stop] 上のセル（提出用のチェックポイント）を先に実行してください"; exit 1; }
V=$P/outputs/videos; mkdir -p $V
BASE=$V/front_policy_final_$FINAL_TAG
CK=logs/rsl_rl/g1_tracking/$FINAL_RUN/$FINAL_CKPT
# 外部コマンドはすべて標準入力を切る（< /dev/null、ffmpeg は -nostdin）。すでにある動画は作り直さない

# (1) ゴーストあり（比較用）→ (2) ゴーストなし（提出用）。RAM を使うので順番に
for MODE in ghost noghost; do
  if [ $MODE = ghost ]; then OUT=$BASE.mp4; OPT=""; else OUT=${BASE}_noghost.mp4; OPT="--no-ghost"; fi
  if [ -f $OUT ]; then echo "[skip] $OUT はすでにあります"; continue; fi
  echo "[render] $MODE → $OUT"
  $PY $P/scripts/render_policy.py --task $TASK --motion-file src/assets/motions/g1/$NAME.npz \
    --checkpoint $CK --friction 0.8 --azimuth $AZ --elevation $EL --distance $DIST $OPT --out $OUT < /dev/null \
    | grep -E '"success"|"mean_joint_err_rad"|"mean_body_pos_err_rel_m"' || true
done

# (3) 3 画面の比較動画：ゴーストあり／なし
for PAIR in "$BASE.mp4:$V/comparison_final_${FINAL_TAG}_front_all.mp4" "${BASE}_noghost.mp4:$V/comparison_final_${FINAL_TAG}_front_all_noghost.mp4"; do
  SRC=${PAIR%%:*}; DST=${PAIR#*:}
  if [ -f $DST ]; then echo "[skip] $DST はすでにあります"; continue; fi
  START=0 bash $P/scripts/make_comparison.sh $HUMAN $REF $SRC $AUDIO $DST < /dev/null
done

# (4) ロボットだけの 1 画面＋元動画の音声（映像は再エンコードしない）
SOLO=$V/final_robot_only_${FINAL_TAG}_noghost_audio.mp4
if [ -f $SOLO ]; then
  echo "[skip] $SOLO はすでにあります"
else
  ffmpeg -nostdin -y -v error -i ${BASE}_noghost.mp4 -i $AUDIO -map 0:v -map 1:a? -c:v copy -c:a aac -shortest $SOLO
fi
ls -la $BASE.mp4 ${BASE}_noghost.mp4 $V/comparison_final_${FINAL_TAG}_front_all.mp4 $V/comparison_final_${FINAL_TAG}_front_all_noghost.mp4 $SOLO

In [ ]:
# ゴーストの有無で物理が変わっていないことの確認（評価値が一致するはず。小数第 4 位の揺れは GPU の計算順）
base = f"{P}/outputs/videos/front_policy_final_{FINAL_TAG}"
a, b = json.load(open(base + '.json')), json.load(open(base + '_noghost.json'))
for k in ['success', 'first_fall_s', 'mean_joint_err_rad', 'mean_body_pos_err_rel_m', 'max_root_xy_drift_m']:
    print(f"{k:<26} ゴーストあり {str(a[k]):<8} なし {b[k]}")

In [ ]:
# 確認：比較用（ゴーストあり）と提出用（ゴーストなし）。どちらも約 12 MB 以上をノートブックに埋め込むので重い
display(show(f"{P}/outputs/videos/comparison_final_{FINAL_TAG}_front_all.mp4", width=1100))
display(show(f"{P}/outputs/videos/comparison_final_{FINAL_TAG}_front_all_noghost.mp4", width=1100))

## 付録：結果を手元に持ち帰るもの

`outputs/eval/nb5/`（`screen_nb4.csv`、`final_compare.csv`、`final_segments_mu0.8.csv`、`best_nb4.txt`、`final_choice.txt`、診断の CSV と動画）、
`outputs/report_figs/fig5_*.png`・`fig6_*.png`、`logs/train_the_feels_full_g25s_ft3_rob.log`。
これらをダウンロードして手元の `pai_final/` の同じ場所に置けば、ノウハウ集の更新とレポートの図に使える。